# KishoLens ETL Pipeline Prototype

This notebook implements a streamed ETL pipeline to ingest the `NilanE/ParallelFiction-Ja_En-100k` dataset from Hugging Face, clean the raw text (removing HTML tags, removing translator notes, and handling Japanese ruby tags), save it to SQLite (`data/kisholens.db`), and preview NLP feature extraction.

In [ ]:
import os
import re
from typing import Optional
from bs4 import BeautifulSoup
from datasets import load_dataset
from sqlmodel import SQLModel, Field, Session, create_engine, select

In [ ]:
# Declare SQLModel tables

class Novel(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    author: str
    source: str

class Chapter(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    novel_id: int = Field(foreign_key="novel.id")
    chapter_number: int
    title: str
    text_ja: str
    text_en: str

In [ ]:
# Text cleaning and parsing methods

def clean_html(text: str) -> str:
    """Removes HTML tags using BeautifulSoup."""
    if not text:
        return ""
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

def clean_japanese(text: str) -> str:
    """Removes Japanese ruby tags (｜ and 《 》)."""
    if not text:
        return ""
    text = re.sub(r'《.*?》', '', text)
    text = text.replace('｜', '')
    return text

def clean_english(text: str) -> str:
    """Strips translator/editor notes matching [TL note: ...] or [T/N: ...]."""
    if not text:
        return ""
    pattern = r'\[(?:TL\s*note|T/N|Editor\'s\s*note|EN|TN):.*?\]'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()

def extract_chapter_info(src: str, trg: str, index: int):
    """
    Extracts chapter number and titles from the first line of raw text.
    If the first line represents a title, it strips it from the returned body text.
    """
    src_first_line = src.strip().split('\n')[0]
    trg_first_line = trg.strip().split('\n')[0]
    
    chapter_number = None
    # Match Japanese headers like "77.素人の気づき"
    match_ja = re.match(r'^(\d+)[.．\s]', src_first_line)
    if match_ja:
        chapter_number = int(match_ja.group(1))
    else:
        # Match English headers like "Chapter 77: ..." or "Chapter 2 - ..." or "hapter 28: ..."
        match_en = re.search(r'(?:[Cc]?hapter|[Cc]h)\s*(\d+)', trg_first_line, re.IGNORECASE)
        if match_en:
            chapter_number = int(match_en.group(1))
        else:
            match_en_start = re.match(r'^(\d+)[.:\s\-]', trg_first_line)
            if match_en_start:
                chapter_number = int(match_en_start.group(1))
    
    is_header = True
    if chapter_number is None:
        chapter_number = index + 1
        is_header = False
        
    title_ja = src_first_line
    if match_ja:
        title_ja = src_first_line[match_ja.end():].strip()
    
    title_en = trg_first_line
    match_en_title = re.match(r'^(?:[Cc]?hapter|[Cc]h)\s*\d+[\s:.\-]*', trg_first_line, re.IGNORECASE)
    if match_en_title:
        title_en = trg_first_line[match_en_title.end():].strip()
    else:
        match_en_start_num = re.match(r'^\d+[\s:.\-]*', trg_first_line)
        if match_en_start_num:
            title_en = trg_first_line[match_en_start_num.end():].strip()
            
    if not is_header:
        title_ja = f"Chapter {chapter_number}"
        title_en = f"Chapter {chapter_number}"
        
    body_ja = src
    body_en = trg
    if is_header:
        body_ja = "\n".join(src.strip().split('\n')[1:])
        body_en = "\n".join(trg.strip().split('\n')[1:])
        
    return chapter_number, title_ja, title_en, body_ja, body_en

In [ ]:
# Baseline NLP Feature Extractor

def extract_features(text: str, lang: str = "en"):
    """
    Computes baseline features: token counts, sentence counts,
    punctuation density, and dialogue ratios.
    """
    if not text:
        return {"token_count": 0, "sentence_count": 0, "punctuation_density": 0.0, "dialogue_ratio": 0.0}
        
    if lang == "en":
        # Tokens (words)
        tokens = re.findall(r'\b\w+\b', text)
        token_count = len(tokens)
        
        # Sentences split by delimiters
        sentences = re.split(r'[.!?]+', text)
        sentences = [s for s in sentences if s.strip()]
        sentence_count = len(sentences)
        
        # Punctuation density
        punctuations = re.findall(r'[.,\/#!$%\^&\*;:{}=\-_`~()?"\']', text)
        punc_count = len(punctuations)
        char_count = len(text)
        punc_density = punc_count / char_count if char_count > 0 else 0.0
        
        # Dialogue ratio (lines starting with quotes)
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        dialogue_lines = [line for line in lines if line.startswith('"') or line.startswith("'") or line.startswith('“') or line.startswith('”')]
        dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    else: # ja
        # Characters as tokens for Japanese baseline
        tokens = [c for c in text if c.strip()]
        token_count = len(tokens)
        
        # Sentences split by delimiters
        sentences = re.split(r'[。！？]+', text)
        sentences = [s for s in sentences if s.strip()]
        sentence_count = len(sentences)
        
        # Punctuation density
        punctuations = re.findall(r'[、。！？「」『』（）―…ー・]', text)
        punc_count = len(punctuations)
        char_count = len(text)
        punc_density = punc_count / char_count if char_count > 0 else 0.0
        
        # Dialogue ratio (lines starting with Japanese open quotes 「 or 『)
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        dialogue_lines = [line for line in lines if line.startswith('「') or line.startswith('『')]
        dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
        
    return {
        "token_count": token_count,
        "sentence_count": sentence_count,
        "punctuation_density": punc_density,
        "dialogue_ratio": dialogue_ratio
    }

In [ ]:
# ETL Orchestration

def run_etl():
    # Resolve path to database to support running from both root and notebooks/ dirs
    if os.path.exists("data"):
        db_path = "data/kisholens.db"
    elif os.path.exists("../data"):
        db_path = "../data/kisholens.db"
    else:
        os.makedirs("../data", exist_ok=True)
        db_path = "../data/kisholens.db"
    
    # Create engine and tables
    engine = create_engine(f"sqlite:///{db_path}")
    SQLModel.metadata.create_all(engine)
    
    print(f"Database path: {db_path}")
    print("Loading ParallelFiction-Ja_En-100k in streaming mode...")
    dataset = load_dataset("NilanE/ParallelFiction-Ja_En-100k", split="train", streaming=True)
    iterator = iter(dataset)
    
    novels_cache = {}
    
    with Session(engine) as session:
        for idx in range(20):
            item = next(iterator)
            meta = item.get('meta', {})
            
            series_title_eng = meta.get('general', {}).get('series_title_eng', 'Unknown')
            series_title_jap = meta.get('general', {}).get('series_title_jap', 'Unknown')
            writer = meta.get('syosetu', {}).get('writer', 'Unknown')
            
            novel_key = (series_title_eng, writer)
            if novel_key not in novels_cache:
                # Check if novel already exists in database
                statement = select(Novel).where(Novel.title == series_title_eng, Novel.author == writer)
                existing_novel = session.exec(statement).first()
                if existing_novel:
                    novels_cache[novel_key] = existing_novel.id
                else:
                    novel = Novel(
                        title=series_title_eng,
                        author=writer,
                        source="syosetu" if "syosetu" in meta else "unknown"
                    )
                    session.add(novel)
                    session.commit()
                    session.refresh(novel)
                    novels_cache[novel_key] = novel.id
                    print(f"Added Novel: '{series_title_eng}' by {writer} (ID: {novel.id})")
                
            novel_id = novels_cache[novel_key]
            
            # Extract and clean chapter text
            chapter_number, title_ja, title_en, body_ja, body_en = extract_chapter_info(item['src'], item['trg'], idx)
            cleaned_ja = clean_japanese(clean_html(body_ja))
            cleaned_en = clean_english(clean_html(body_en))
            
            # Check if this chapter already exists
            statement_ch = select(Chapter).where(Chapter.novel_id == novel_id, Chapter.chapter_number == chapter_number)
            existing_chapter = session.exec(statement_ch).first()
            if not existing_chapter:
                chapter = Chapter(
                    novel_id=novel_id,
                    chapter_number=chapter_number,
                    title=title_en,
                    text_ja=cleaned_ja,
                    text_en=cleaned_en
                )
                session.add(chapter)
                session.commit()
                print(f"  Ingested Chapter {chapter_number}: {title_en}")
            else:
                print(f"  Chapter {chapter_number} already ingested. Skipping DB insertion.")
            
            # Print baseline features demo
            feat_ja = extract_features(cleaned_ja, lang="ja")
            feat_en = extract_features(cleaned_en, lang="en")
            print(f"    Features (JA): Tokens={feat_ja['token_count']}, Sentences={feat_ja['sentence_count']}, PuncDensity={feat_ja['punctuation_density']:.3f}, DialogueRatio={feat_ja['dialogue_ratio']:.3f}")
            print(f"    Features (EN): Tokens={feat_en['token_count']}, Sentences={feat_en['sentence_count']}, PuncDensity={feat_en['punctuation_density']:.3f}, DialogueRatio={feat_en['dialogue_ratio']:.3f}")
            
    # Dispose of engine to release lock
    engine.dispose()
    print("\nETL run completed successfully.")

run_etl()

In [ ]:
# Verification Query

# Resolve path to database
if os.path.exists("data/kisholens.db"):
    db_path = "data/kisholens.db"
else:
    db_path = "../data/kisholens.db"

engine = create_engine(f"sqlite:///{db_path}")
with Session(engine) as session:
    novels = session.exec(select(Novel)).all()
    print(f"Total novels in database: {len(novels)}")
    for n in novels:
        print(f"  - [{n.id}] {n.title} (by {n.author})")
        
    chapters = session.exec(select(Chapter).limit(5)).all()
    print(f"\nFirst 5 chapters:")
    for c in chapters:
        print(f"  - [{c.id}] Chapter {c.chapter_number}: {c.title} (Novel ID: {c.novel_id})")
        print(f"    JA text (first 100 chars): {c.text_ja[:100]}...")
        print(f"    EN text (first 100 chars): {c.text_en[:100]}...")
engine.dispose()